# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 37.3 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sample_physics.xlsx')

In [ ]:
data.head()

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,5148,https://drive.google.com/drive/folders/14FKkR6...,Palestine,STEM,Physics,High,مقاومتان مقدار كل منهما م إذا وصلتا على التو...,NaN,A,0.5 م,م,2 م,1.5 م,NaN,0
1,4956,exams-28a8e93b-7722-11ea-9116-54bef70b159e,NaN,STEM,Physics,High,ما هو مدى الموجات الصوت ةٌ الت تٌمكن الإنسان ا...,NaN,D,أكبر من Hz 20000,من Hz 10 إلى Hz 10000,أقل من Hz 20,من Hz 20 إلى Hz 20000,NaN,0
2,5037,https://drive.google.com/drive/folders/14FKkR6...,Palestine,STEM,Physics,High,جسم كتلته 5 كغم وكمية تحركه 15 كغم.م/ث ، فإن ...,NaN,A,1.7 نيوتن,0.35 نيوتن,5.1 نيوتن,11 نيوتن,NaN,0
3,5164,https://drive.google.com/drive/folders/14FKkR6...,Palestine,STEM,Physics,High,الفرق في درجات الحرارة الالزمة لمادة معاملها ا...,NaN,D,20 كلفن,5 كلفن,12 كلفن,10 كلفن,NaN,0
4,5075,https://drive.google.com/drive/folders/14FKkR6...,Palestine,STEM,Physics,High,كمية التحرك للنظام الذي يتكون من كرتين متماثل...,NaN,B,2 ك ع,صفر,ك ع,0.5 ك ع,NaN,0


# Zero Shot

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة
'''
zero_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
) 0.5 م,2
) من Hz 20 إلى Hz 20000,1
) 0.35 نيوتن\n\nالثبات في الحركة يعني أن القوة الصافية المؤثرة على الجسم تساوي صفر. لكن السؤال يتعلق بتغيير السرعة، وليس الثبات. ومع ذلك، بناءً على المعلومات المقدمة، يمكننا حساب التسارع ثم القوة باستخدام قانون نيوتن الثاني للحركة (F = ma).\n\nالسرعة الأولية = 15 م/ث\nالسرعة النهائية = 8 م/ث\nالزمن = 15 ثانية\n\nالتسارع (a) = (السرعة النهائية - السرعة الأولية) / الزمن\na = (8 - 15) / 15 = -7/15 م/ث^2\n\nالقوة (F) = الكتلة × التسارع\nF = 5 كغم × (-7/15) م/ث^2 = -35/15 نيوتن ≈ -2.33 نيوتن\n\nلكن بما أن الخيارات لا تتضمن قيمة سالبة، فإن القيمة الأقرب من حيث الحجم هي 2.33 نيوتن، لكن الخيار الأقرب في الخيارات المعطاة هو 2.5 نيوتن، لكن لا يوجد هذا الخيار. الخيار الأقرب في الخيارات المعطاة هو 2.35 نيوتن، لكن لا يوجد هذا الخيار أيضاً. ومع ذلك، بناءً على الخيارات المعطاة، فإن الخيار الأقرب هو 2.33 نيوتن، لكن لا يوجد هذا الخيار. يبدو أن هناك خطأ في الخيارات المعطاة. ولكن بناءً على الخيارات المعطاة، فإن الخيار الأقرب هو 2.33 نيوتن، لكن لا يوجد هذا الخيار. ومع ذلك، إذا كنا ننظر إلى الخيارات المعطاة، فإن الخيار الأقرب هو 0.35 نيوتن، لكن هذا لا يبدو صحيحاً بناءً على الحسابات. قد يكون هناك خطأ في السؤال أو الخيارات. ولكن بناءً على الخيارات المعطاة، فإن الخيار الأقرب هو 0.35 نيوتن، لكن هذا لا يبدو صحيحاً بناءً على الحسابات. ومع ذلك، إذا كنا ننظر إلى الخيارات المعطاة، فإن الخيار الأقرب هو 0.35 نيوتن، لكن هذا لا يبدو صحيحاً بناءً على الحسابات.,1
) 20 كلفن\n\nالثابت الذي ذكر في السؤال هو معامل التوصيل الحراري وليس معامل التغير في المقاومة مع درجة الحرارة. ومع ذلك، بناءً على قانون ميترش، الذي ينص على أن المقاومة الكهربائية لمادة تتغير بشكل خطي مع درجة الحرارة، يمكننا استخدام معامل التغير في المقاومة مع درجة الحرارة (α). لكن بما أن هذا المعامل غير متوفر في السؤال، فإن المعلومات المقدمة غير كافية لحساب الفرق الدقيق في درجة الحرارة. لكن إذا افترضنا أن المعامل المذكور هو معامل التغير في المقاومة مع درجة الحرارة (α)، فإن الصيغة ستكون:\n\nR2 = R1 * (1 + α * ΔT)\n\nحيث R2 هي المقاومة بعد التغير في درجة الحرارة، R1 هي المقاومة الأولية، ΔT هو الفرق في درجة الحرارة.\n\nإذا أردنا أن تنخفض المقاومة للنصف، R2 = R1/2.\n\nلذلك،\n\nR1/2 = R1 * (1 + α * ΔT)\n\nبالتبسيط،\n\n1/2 = 1 + α * ΔT\n\nΔT = (1/2 - 1) / α = -1/2α\n\nباستخدام α = -0.05/K (لاحظ أن المعامل السلبي يشير إلى أن المقاومة تقل مع زيادة درجة الحرارة)،\n\nΔT = -1/2 * (-0.05/K) = 0.025/K\n\nلكن هذا لا يطابق الخيارات المعطاة. قد يكون هناك خطأ في السؤال أو في الخيارات. ولكن بناءً على الخيارات المعطاة، الخيار (أ) 20 كلفن هو الأقرب إلى النتيجة التي يمكن الحصول عليها إذا تم تفسير المعامل بشكل مختلف أو إذا كانت هناك أخطاء في الحساب. لكن بشكل دقيق، لا يمكن تحديد الإجابة بناءً على المعلومات المقدمة.\n\nومع ذلك، إذا افترضنا أن السؤال يتعلق بمعامل التغير في المقاومة مع درجة الحرارة (α) وليس معامل التوصيل الحراري، فإن الإجابة الأقرب من الخيارات المعطاة هي (أ) 20 كلفن، ولكن هذا يعتمد على افتراضات إضاف,1
) صفر\n\nالثبات الكلي للنظام هو مجموع ثبات كل جسم. بما أن الكرتين متماثلتين في الكتلة وتتحركان في اتجاهين متعاكسين بنفس السرعة، فإن ثبات كل منهما سيكون مساوياً ولكن في اتجاهين متعاكسين. لذلك، عندما نجمعهما، فإن الثبات الكلي للنظام سيكون صفر.,1
...,...
) 20 كغم.م/ث,1
) نصف القطر\n\nالثابت أن شدة المجال المغناطيسي عند مركز ملف دائري تتناسب عكسياً مع نصف القطر.,1
مساو لتسارع الرصاصة\n\n(أ) الإجابة,1


In [ ]:
pred_zero['question'] = data['Question']
pred_zero['Option 1'] = data['Option 1']
pred_zero['Option 2'] = data['Option 2']
pred_zero['Option 3'] = data['Option 3']
pred_zero['Option 4'] = data['Option 4']
pred_zero.to_excel('Fanar-QA-Physics-ZeroShot.xlsx', index = False)

In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_zero['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.5556    0.4082    0.4706        49
           B     0.6154    0.8276    0.7059        58
           C     0.5897    0.5476    0.5679        42
           D     0.6170    0.5686    0.5918        51

    accuracy                         0.6000       200
   macro avg     0.5944    0.5880    0.5841       200
weighted avg     0.5958    0.6000    0.5902       200



# Pred Few Shot

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

الإجابة:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

الإجابة:
ب) مستقبلات

مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

الإجابة:
أ) ناتج الطرح
'''

few_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
) على التوازي,2
2 ك ع,2
) صفر,2
) أوم,2
) 20 كلفن\n\nالثابت الذي ذكر في السؤال هو معامل التوصيل الحراري وليس معامل التمدد الحراري. لكن بناءً على سياق السؤال، يبدو أنه يشير إلى معامل التمدد الحراري. معامل التمدد الحراري الخطي (α) عادة ما يستخدم لحساب التغير في الطول مع تغير درجة الحرارة. لكن في سياق مقاومة المادة، يمكن استخدام معامل التوصيل الحراري (β) الذي يصف التغير في المقاومة مع تغير درجة الحرارة. ومع ذلك، فإن الصيغة الأكثر شيوعاً المستخدمة هي:\n\nR = R₀ (1 + αΔT)\n\nحيث R₀ هي المقاومة الأولية، R هي المقاومة النهائية، α هو معامل التمدد الحراري، و ΔT هو التغير في درجة الحرارة.\n\nإذا كانت المقاومة تقل إلى النصف، فإن R = R₀/2.\n\nلذلك، R₀ (1 + αΔT) = R₀/2\n\nبالتبسيط، نحصل على:\n\n1 + αΔT = 1/2\n\nαΔT = 1/2 - 1 = -1/2\n\nΔT = -1/(2α)\n\nإذا كانت α = -0.05/K (وهو معامل التمدد الحراري السلبي، وهو غير شائع ولكن يمكن أن يحدث في بعض المواد)، فإن:\n\nΔT = -1/(2 * -0.05/K) = 10 K\n\nلكن يجب أن نلاحظ أن معامل التوصيل الحراري عادة ما يكون إيجابياً، مما يعني أن المقاومة تزداد مع زيادة درجة الحرارة. ومع ذلك، بناءً على الخيارات المعطاة، فإن الإجابة الأقرب هي:\n\na) 20 كلفن\n\nلكن هذا غير صحيح بناءً على المعادلات. إذا افترضنا أن معامل التمدد الحراري هو إيجابي، فإن ΔT سيكون سالب، مما يعني أن درجة الحرارة يجب أن تنخفض. لكن الخيارات لا تحتوي على خيار سالب. بناءً على الخيارات المعطاة، لا يوجد خيار صحيح تماماً. ولكن إذا كنا نضطر لاختيار واحد، بناءً على المعادلة، فإن الإجابة الأقرب ستكون:\n\nd),1
...,...
) 20 كغم.م/ث,1
) نصف القطر,1
مساو لتسارع الرصاصة\n\n(أ),1


In [ ]:
pred_few['question'] = data['Question']
pred_few['Option 1'] = data['Option 1']
pred_few['Option 2'] = data['Option 2']
pred_few['Option 3'] = data['Option 3']
pred_few['Option 4'] = data['Option 4']
pred_few.to_excel('Fanar-QA-Physics-FewShot.xlsx', index = False)

In [ ]:
pred_few['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
B,62
D,51
C,47
A,38
Unclassified,2


In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_few['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.6053    0.4694    0.5287        49
           B     0.6935    0.7414    0.7167        58
           C     0.5532    0.6190    0.5843        42
           D     0.5882    0.5882    0.5882        51
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6100       200
   macro avg     0.4880    0.4836    0.4836       200
weighted avg     0.6156    0.6100    0.6101       200



# CoT

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

اتبع الخطوات التالية للوصول إلى الإجابة الصحيحة:

الخطوة 1: فهم السؤال
اقرأ السؤال بعناية لفهم ما يُطلب منك تحديداً. حاول فهم الحاجة والسياق الكامن خلف السؤال.

الخطوة 2: تحليل الخيارات
اقرأ كل خيار بعناية. فكّر في معنى كل خيار وكيف يرتبط بالسؤال.

الخطوة 3: تقييم الخيارات
استخدم معرفتك ومعلوماتك ذات الصلة لتقييم كل خيار. استبعد الخيارات الخاطئة بشكل واضح، ثم قارن بين الخيارات المتبقية لتحديد الأنسب.

الخطوة 4: اختيار أفضل إجابة
اختر الخيار الأكثر منطقية وصحة بناءً على التقييم في الخطوات السابقة.

الخطوة 5: كتابة الإجابة
اكتب الإجابة الصحيحة فقط باستخدام هذا التنسيق الصارم:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

التحليل:
- الخطوة 1: فهم السؤال
المطلوب حساب المقاومة الكهربائية لمصباح قدرته 100 واط وجهده 220 فولت.
- الخطوة 2: تحليل الخيارات
لحساب المقاومة نستخدم الجهد والقدرة الكهربائية المعطاة. الخيارات هي 220 أوم، 202 أوم، 100 أوم، و484 أوم.
- الخطوة 3: تقييم الخيارات
نستخدم العلاقة التالية لحساب المقاومة:
R = V^2/P
نعوض القيم المعطاة:
R = 220^2/100 = 48400/100 = 484 أوم
- الخطوة 4: اختيار الإجابة الصحيحة
من بين الخيارات، 484 أوم هي الإجابة الصحيحة، وهي الخيار د.

الإجابة النهائية:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

التحليل:
- الخطوة 1: فهم السؤال
السؤال يطلب: أي من الخيارات التالية يتشابه عمله مع عمل كرات الدم البيضاء في جسم الإنسان؟
كرات الدم البيضاء هي جزء من جهاز المناعة؛ وظيفتها الأساسية التعرف على مسببات الأمراض (بكتيريا، فيروسات، إلخ) ومهاجمتها.
- الخطوة 2: تحليل الخيارات
أ) جلوكوزيدات: مركبات سكرية؛ ليست جزءاً من جهاز المناعة.
ب) مستقبلات: المستقبلات تلعب دوراً مهماً في التعرف على مسببات الأمراض وإرسال إشارات لتحفيز الاستجابة المناعية.
ج) أحماض أمينية غير بروتينية: مركبات نادرة؛ ليست مرتبطة مباشرة بوظيفة جهاز المناعة.
د) سيفالوسبورين: مضاد حيوي يقضي على البكتيريا لكنه ليس مكوناً خلوياً في جهاز المناعة.
- الخطوة 3: تقييم الخيارات
ما الذي يشبه وظيفة كرات الدم البيضاء — أي القدرة على التعرف على مسببات الأمراض والاستجابة لها؟
المستقبلات تلعب دوراً رئيسياً في التعرف على الميكروبات وتفعيل الاستجابات المناعية.
- الخطوة 4: اختيار الإجابة الصحيحة
الخيار الأقرب لوظيفة كرات الدم البيضاء هو المستقبلات.

الإجابة النهائية:
ب) مستقبلات


مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

التحليل:
- الخطوة 1: فهم السؤال
السؤال هو: بماذا يسمى العدد 50 في عملية الطرح 70 – 20 = 50 ؟
في عملية الطرح:
العدد الأول (70) يسمى المطروح منه.
العدد الثاني (20) يسمى المطروح.
الناتج (50) يسمى ناتج الطرح أو الفرق.
- الخطوة 2: تحليل الخيارات
ناتج الطرح: هو العدد الناتج من عملية الطرح — وهو 50 في هذه الحالة.
المطروح: هو العدد الذي يتم طرحه — أي 20، وليس 50.
المطروح منه: هو العدد الذي نطرح منه — أي 70، وليس 50.
لاشيء مما سبق: غير صحيح لأن الخيار (أ) صحيح.
- الخطوة 3: تقييم الخيارات
العدد 50 هو ناتج عملية الطرح.
- الخطوة 4: اختيار الإجابة الصحيحة
الإجابة الصحيحة هي ناتج الطرح.

الإجابة النهائية:
أ) ناتج الطرح
'''

cot_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
/\nالتحليل:\n- الخطوة 1: فهم السؤال\nالسؤال يطلب المقاومة المكافئة لمقاومتين متصلتين بالتوازي، كل منهما بمقاومة م.\n- الخطوة 2: تحليل الخيارات\nالخيارات هي 0.5 م، م، 2 م، و1.5 م.\n- الخطوة 3: تقييم الخيارات\nعند توصيل مقاومتين بالتوازي، يتم حساب المقاومة المكافئة باستخدام الصيغة:\n1/R_total = 1/R1 + 1/R2\nحيث R1 و R2 هما المقاومتان.\nبما أن كل مقاومة تساوي م، فإن:\n1/R_total = 1/م + 1/م = 2/م\nR_total = م/2 = 0.5 م\n- الخطوة 4: اختيار الإجابة الصحيحة\nالمقاومة المكافئة هي 0.5 م، وهي الخيار (أ).\n\nالإجابة النهائية:\nأ) 0.5 م,1
) من Hz 20 إلى Hz 20000,1
= (م * Δv) / Δt\nحيث:\nu = مقدار القوة\nm = الكتلة = 5 كغم\nΔv = التغير في السرعة = 8 م/ث - 0 م/ث = 8 م/ث (بما أن السرعة الأولية غير معطاة، نفترض أنها 0 م/ث)\nΔt = الزمن = 15 ثانية\n\nu = (5 كغم * 8 م/ث) / 15 ثانية\nu = 40 كغم.م/ث² / 15 ثانية\nu = 2.666... نيوتن\n\nبما أن الخيارات لا تحتوي على 2.666... نيوتن، ولكن الأقرب هو:\n\nالإجابة:\nc) 5.1 نيوتن (وهو تقريب لـ 2.666... نيوتن)\n\nلكن يجب ملاحظة أن الخيار c) 5.1 نيوتن ليس دقيقاً تماماً بناءً على الحسابات، ولكنها الأقرب بين الخيارات المعطاة. ومع ذلك، إذا كانت الخيارات أكثر دقة، فإن الإجابة الصحيحة ستكون حوالي 2.67 نيوتن.,1
/\nلتحديد الفرق في درجات الحرارة اللازمة لمادة معاملها الحراري -0.05/K لإنقاص مقاومتها للنصف، يمكننا استخدام الصيغة التالية:\n\nR = R₀ * e^(α * ΔT)\n\nحيث:\nR = المقاومة النهائية (نصف المقاومة الأصلية)\nR₀ = المقاومة الأصلية\nα = معامل الحرارة (في هذه الحالة، -0.05/K)\nΔT = الفرق في درجات الحرارة\n\nنريد أن نجد ΔT عندما R = R₀/2.\n\nR₀/2 = R₀ * e^(-0.05 * ΔT)\n\nنقسم كلا الطرفين على R₀:\n\n1/2 = e^(-0.05 * ΔT)\n\nنأخذ اللوغاريتم الطبيعي لكلا الطرفين:\n\nln(1/2) = -0.05 * ΔT\n\nΔT = ln(1/2) / -0.05\n\nΔT ≈ 13.86 كلفن\n\nبينما الخيارات المعطاة لا تتطابق تماماً مع النتيجة الدقيقة، الخيار الأقرب هو:\n\nc) 12 كلفن\n\nلكن يجب ملاحظة أن النتيجة الدقيقة هي حوالي 13.86 كلفن، لذا لا يوجد خيار صحيح تماماً بين الخيارات المعطاة. ولكن بناءً على الخيارات المتاحة، الخيار c) 12 كلفن هو الأقرب. ومع ذلك، في سياق أكثر دقة، يجب أن يكون الجواب 13.86 كلفن.\n\nولكن بناءً على الخيارات المعطاة، سأختار:\n\nc) 12 كلفن (على الرغم من أن هذا ليس دقيقاً تماماً),1
"/\nالتحليل:\n- الخطوة 1: فهم السؤال\nالسؤال يطلب كمية التحرك للنظام الذي يتكون من كرتين متماثلتين في الكتلة ""ك"" وتسيران باتجاهين متعاكسين بنفس السرعة ""ع"".\n- الخطوة 2: تحليل الخيارات\nكمية التحرك (الزخم) تُحسب بضرب الكتلة في السرعة.\n- الخطوة 3: تقييم الخيارات\nلكل كرة، كمية التحرك هي ك ع. بما أن الكرتين تتحركان باتجاهين متعاكسين، فإن كمية التحرك الكلية للنظام ستكون مجموع كمية التحرك لكلا الكرتين. ومع ذلك، بما أن الكرتين تتحركان باتجاهين متعاكسين، فإن كمية التحرك ستكون صفر لأن كمية التحرك هي كمية متجهة (تحتاج إلى اتجاه).\n- الخطوة 4: اختيار الإجابة الصحيحة\nبما أن كمية التحرك الكلية للنظام هي صفر، فإن الإجابة الصحيحة هي الخيار ب.\n\nالإجابة النهائية:\nب) صفر",1
...,...
\nالخطوة 1: فهم السؤال\nالسؤال يطلب تحديد الكتلة عندما تضاعف الطاقة الحركية لجسم يتحرك بسرعة 10 كغم.م/ث.\n\nالخطوة 2: تحليل الخيارات\nالخيارات هي a) 5 كغم.م/ث، b) 20 كغم.م/ث، c) 10 كغم.م/ث، d) 10 كغم.م/ث.\n\nالخطوة 3: تقييم الخيارات\nالطاقه الحركية (KE) لجسم معين تعطى بالصيغة: KE = (1/2) * m * v^2، حيث m هي الكتلة وv هي السرعة.\n\nإذا ضاعفنا الطاقة الحركية، فإن المعادلة الجديدة ستكون: 2 * KE = (1/2) * m * v^2.\n\nبما أن السرعة لا تتغير، يمكننا إعادة ترتيب المعادلة لإيجاد الكتلة الجديدة (m'):\n\n2 * KE = (1/2) * m' * v^2\n4 * KE = m' * v^2\nm' = 4 * KE / v^2\n\nبما أن KE = (1/2) * m * v^2، يمكننا استبدال KE في المعادلة:\n\nm' = 4 * (1/2) * m * v^2 / v^2\nm' = 2 * m\n\nإذا كانت الكتلة الأصلية m = 10 كغم، فإن الكتلة الجديدة m' = 2 * 10 كغم = 20 كغم.\n\nالخطوة 4: اختيار الإجابة الصحيحة\nبما أن الكتلة الجديدة m' = 20 كغم، فإن الإجابة الصحيحة هي b) 20 كغم.م/ث.\n\nالإجابة النهائية:\nب) 20 كغم.م/ث,1
) نصف القطر\n\nالتفسير:\nتتناسب شدة المجال المغناطيسي عند مركز ملف دائري عكسيا مع نصف القطر (r) وفقا لقانون أمبير. حيث أن شدة المجال المغناطيسي (B) تتناسب عكسيا مع نصف القطر (r) عندما يكون التيار (I) ثابت. لذلك، كلما زاد نصف القطر، تقل شدة المجال المغناطيسي عند المركز.,1
) مساو لتسارع الرصاصة\n\nالتفسير: وف

In [ ]:
pred_cot['question'] = data['Question']
pred_cot['Option 1'] = data['Option 1']
pred_cot['Option 2'] = data['Option 2']
pred_cot['Option 3'] = data['Option 3']
pred_cot['Option 4'] = data['Option 4']
pred_cot.to_excel('Fanar-QA-Physics-CoT.xlsx', index = False)

In [ ]:
pred_cot['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
B,63
C,48
D,44
A,35
Unclassified,9
A/C,1


In [ ]:
print(classification_report(y_true, pred_cot['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.6857    0.4898    0.5714        49
         A/C     0.0000    0.0000    0.0000         0
           B     0.6825    0.7414    0.7107        58
           C     0.5417    0.6190    0.5778        42
           D     0.7273    0.6275    0.6737        51
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6250       200
   macro avg     0.4395    0.4129    0.4223       200
weighted avg     0.6651    0.6250    0.6392       200

